# Optimiize Onnx Models
### .onnx (onnxruntime-gpu)
Aug 7th

### About

NanoSAM is a Segment Anything (SAM) model variant that is capable of running in 🔥 real-time 🔥 on NVIDIA Jetson Orin Platforms with NVIDIA TensorRT.

## Optimization Steps

onnxoptimizer

```
python -m onnxoptimizer input_model.onnx output_model.onnx
```

Model for quantization by inferring shapes and sometimes performing basic graph optimizations. Quantize_dynamic() and other onnxruntime.quantization functions are for quantization. Dynamic quantization computes parameters at runtime, while static quantization (Post-Training Quantization - PTQ) requires calibration data.

In [1]:
"""
# clean env wipe & activate
conda env remove --name onnxoptim_arm64 -y
conda deactivate
conda create -n onnxoptim_arm64 python=3.10 -y
conda activate onnxoptim_arm64

sudo apt update
sudo apt install libcudnn8 libcudnn8-dev
conda install -c conda-forge libstdcxx-ng=12 -y

"""

'\n# clean env wipe & activate\nconda env remove --name onnxoptim_arm64 -y\nconda deactivate\nconda create -n onnxoptim_arm64 python=3.10 -y\nconda activate onnxoptim_arm64\n\nsudo apt update\nsudo apt install libcudnn8 libcudnn8-dev\nconda install -c conda-forge libstdcxx-ng=12 -y\n\n'

In [ ]:
"""
# Installation of onnxoptim_arm64 repo requirements

# More requirements

pip install timm==1.0.17 ultralytics==8.0.120 matplotlib==3.5.3 'opencv-python<4.9' jetson-stats

# Typical np version mismatch

pip uninstall -y numpy
pip install "numpy>=1.19.2,<2.0"
# pip install numpy==1.26.4 


# --- JP61 SAFE TORCH & TORCHVISION ------------- #

# Now install the CUDA PyTorch 2.5.0a0 (compatible with jp61 torch)
pip install --no-cache https://developer.download.nvidia.com/compute/redist/jp/v61/pytorch/torch-2.5.0a0+872d972e41.nv24.08.17622132-cp310-cp310-linux_aarch64.whl

# download torchvision 0.20.0a0 (compatible with jp61 torch)
cd ~/vision/
python3 setup.py install
cd ~

# --- ONNXRUNTIME-GPU (build from whl) ------------- #

pip uninstall onnxruntime -y
pip install https://github.com/ultralytics/assets/releases/download/v0.0.0/onnxruntime_gpu-1.20.0-cp310-cp310-linux_aarch64.whl
pip install sympy==1.13.1 numpy==1.26.4

# --- ONNX OPTIM (build from scratch) ------------- #

pip install onnxoptimizer
pip install build/Linux/Release/dist/onnxruntime_gpu-*.whl --force-reinstall

"""

# Debug Support

# Check your conda env GLIBCXX versions
# strings /home/copter/miniconda3/envs/nanosam_arm64/lib/python3.10/site-packages/zmq/backend/cython/../../../../.././libstdc++.so.6 | grep GLIBCXX

# trt2torch is apparently an issue and is obsolete for trt --version=10.3.0 
# additionally `pip install pycuda`

'\n# Installation of onnxoptim_arm64 repo requirements\n\n# More requirements\n\npip install timm==1.0.17 ultralytics==8.0.120 matplotlib==3.5.3 \'opencv-python<4.9\' jetson-stats\n\n# Typical np version mismatch\n\npip uninstall -y numpy\npip install "numpy>=1.19.2,<2.0"\n# pip install numpy==1.26.4 \n\n\n# --- JP61 SAFE TORCH & TORCHVISION ------------- #\n\n# Now install the CUDA PyTorch 2.5.0a0 (compatible with jp61 torch)\npip install --no-cache https://developer.download.nvidia.com/compute/redist/jp/v61/pytorch/torch-2.5.0a0+872d972e41.nv24.08.17622132-cp310-cp310-linux_aarch64.whl\n\n# download torchvision 0.20.0a0 (compatible with jp61 torch)\ncd ~/vision/\npython3 setup.py install\ncd ~\n\n# --- ONNXRUNTIME-GPU (build from scratch) ------------- #\n\ncd ~/onnxruntime\npip uninstall onnxruntime -y\npip install build/Linux/Release/dist/onnxruntime_gpu-*.whl --force-reinstall\npip install sympy==1.13.1 numpy==1.26.4\n\n# --- ONNXRUNTIME-GPU (build from scratch) ------------- #\n\

In [3]:
# # from nanosam.utils.predictor import Predictor
# import onnxruntime as ort
# import numpy as np
# from PIL import Image, ImageDraw
# import matplotlib.pyplot as plt


## Clean  Graph

pip3 install onnxoptimizer

Optimization is done in the cli

this just cleans the graph, graph-level optimizations on your ONNX model. These are general optimizations like constant folding, dead code elimination, and node fusion that are not specific to any hardware... o

nnxruntime.quantization.preprocess, quantize_dynamic is the next step

In [4]:
# Load the image
"""
python -m onnxoptimizer input_model.onnx output_model.onnx

# python3 -m onnxoptimizer -h                                 
python -m onnxoptimizer /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encode_optim_clean.onnx --fixed_point True
python -m onnxoptimizer /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_clean.onnx --fixed_point True



usage: python -m onnxoptimizer input_model.onnx output_model.onnx 

onnxoptimizer command-line api

optional arguments:
  -h, --help            show this help message and exit
  --print_all_passes    print all available passes
  --print_fuse_elimination_passes
                        print all fuse and elimination passes
  -p [PASSES ...], --passes [PASSES ...]
                        list of optimization passes name, if no set, fuse_and_elimination_passes will be used
  --fixed_point         fixed point

"""

'\npython -m onnxoptimizer input_model.onnx output_model.onnx\n\n# python3 -m onnxoptimizer -h                                 \npython -m onnxoptimizer /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encode_optim_clean.onnx --fixed_point True\npython -m onnxoptimizer /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_clean.onnx --fixed_point True\n\n\n\nusage: python -m onnxoptimizer input_model.onnx output_model.onnx \n\nonnxoptimizer command-line api\n\noptional arguments:\n  -h, --help            show this help message and exit\n  --print_all_passes    print all available passes\n  --print_fuse_elimination_passes\n                        print all fuse and elimination passes\n  -p [PASSES ...], --passes [PASSES ...]\n                        list of optimization passes name, if no set, fuse_and_elimination_passes will be used\n  --f

In [5]:
encoder_model_path="/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx"
decoder_model_path="/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx"

## Onnxruntime to Quantize Models

Source: https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html

In [6]:
# Inspect Onnx models

import onnx
import onnx.numpy_helper

def inspect_model_precision(model_path):
    """
    Inspects an ONNX model to determine the data type (precision) of its initializers (weights).
    """
    try:
        model = onnx.load(model_path)
        print(f"--- Inspecting Model: {model_path} ---")

        initializers = model.graph.initializer
        if not initializers:
            print("No initializers (weights) found in the model.")
            return

        # Get a list of unique data types from the initializers
        data_types = set()
        for initializer in initializers:
            data_type = onnx.helper.tensor_dtype_to_string(initializer.data_type)
            data_types.add(data_type)
        
        # Determine the primary precision based on the most common type
        # For simplicity, we'll just report all unique types found.
        if len(data_types) == 1:
            print(f"✅ Model appears to be a single precision type: {list(data_types)[0]}")
        else:
            print(f"⚠️ Model is a mixed precision type. Found the following data types: {data_types}")

        # You can also inspect the data type of the model's inputs and outputs
        input_type = onnx.helper.tensor_dtype_to_string(model.graph.input[0].type.tensor_type.elem_type)
        print(f"Model input data type: {input_type}")

        print("-" * 50)

    except FileNotFoundError:
        print(f"❌ Error: Model file not found at {model_path}")
    except Exception as e:
        print(f"❌ An error occurred: {e}")

encoder_model_path = "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx"
decoder_model_path = "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx"

inspect_model_precision(encoder_model_path)
inspect_model_precision(decoder_model_path)

--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx ---
✅ Model appears to be a single precision type: TensorProto.FLOAT
Model input data type: TensorProto.FLOAT
--------------------------------------------------
--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx ---
✅ Model appears to be a single precision type: TensorProto.FLOAT
Model input data type: TensorProto.FLOAT
--------------------------------------------------


In [7]:
## ONNX Runtime Website Docs

""" 
python -m onnxruntime.quantization.preprocess --help

"""

' \npython -m onnxruntime.quantization.preprocess --help\n\n'

In [8]:
# # Model Preprocessing: Encoder (only if ding INT8 Quant)

# import onnx
# from onnxruntime.quantization.shape_inference import quant_pre_process

# # Define your input and output paths
# enc_input_model_path = encoder_model_path
# enc_output_model_path = "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_preproc.onnx"

# # Load the model
# try:
#     model = onnx.load(enc_input_model_path)
# except FileNotFoundError:
#     print(f"Error: Model file not found at '{enc_input_model_path}'. Please check the path.")
# except Exception as e:
#     print(f"An error occurred while loading the model: {e}")
    
# # Execute the preprocessing
# # The function will perform symbolic shape inference, model optimization, and ONNX shape inference
# try:
#     quant_pre_process(enc_input_model_path, enc_output_model_path)
#     print(f"✅ Preprocessing successful. Processed model saved to '{enc_output_model_path}'.")
# except Exception as e:
#     print(f"❌ An error occurred during preprocessing: {e}")

In [9]:
# # Model Preprocessing: Decoder (only if ding INT8 Quant)

# import onnx
# from onnxruntime.quantization.shape_inference import quant_pre_process

# # Define your input and output paths
# dec_input_model_path = decoder_model_path
# dec_output_model_path = "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_preproc.onnx"

# # Load the model
# try:
#     model = onnx.load(dec_input_model_path)
# except FileNotFoundError:
#     print(f"Error: Model file not found at '{dec_input_model_path}'. Please check the path.")
# except Exception as e:
#     print(f"An error occurred while loading the model: {e}")
    
# # Execute the preprocessing
# # The function will perform symbolic shape inference, model optimization, and ONNX shape inference
# try:
#     quant_pre_process(dec_input_model_path, dec_output_model_path)
#     print(f"✅ Preprocessing successful. Processed model saved to '{dec_output_model_path}'.")
# except Exception as e:
#     print(f"❌ An error occurred during preprocessing: {e}")

In [20]:
# Set encode paths

import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# for the preprocessed models (FP16 use case)
enc_path = "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx"
dec_path = "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx"

# # for the preprocessed models (INT8 only)
# enc_path = "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_preproc.onnx"
# dec_path = "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_preproc.onnx"


In [21]:
# Encoder dynamnic quant

model_fp32 = enc_path
model_quant = "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_dyn.onnx"
# quantize_dynamic(model_fp32, model_quant)

# quantize_dynamic(model_fp32, model_quant)
quantize_dynamic(
    model_input=model_fp32,
    model_output=model_quant,
    op_types_to_quantize=["MatMul", "Gemm"]
)

In [19]:
# Decoder dynamnic quant

import onnx

# Verify the model first
try:
    model = onnx.load(dec_path)
    onnx.checker.check_model(model)
    print("✅ Model is valid")
except Exception as e:
    print(f"❌ Model validation failed: {e}")

model_fp32 = dec_path
model_quant = "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_dyn.onnx"
# quantize_dynamic(model_fp32, model_quant)
quantize_dynamic(
    model_input=model_fp32,
    model_output=model_quant,
    op_types_to_quantize=["MatMul", "Gemm"]
)

✅ Model is valid


In [22]:
## inspect post dynamic quantization 

inspect_model_precision("/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_dyn.onnx")
inspect_model_precision("/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_dyn.onnx")

--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_dyn.onnx ---
✅ Model appears to be a single precision type: TensorProto.FLOAT
Model input data type: TensorProto.FLOAT
--------------------------------------------------
--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_dyn.onnx ---
⚠️ Model is a mixed precision type. Found the following data types: {'TensorProto.INT8', 'TensorProto.FLOAT'}
Model input data type: TensorProto.FLOAT
--------------------------------------------------


### Alternate quantization

using `onnxconverter_common` for auto-mixed and fp16

In [ ]:
# # MIXED PRECISION
# from onnxconverter_common import auto_mixed_precision
# import onnx


# model = onnx.load("path/to/model.onnx")

# # Assuming x is the input to the model
# feed_dict = {'input': x.numpy()}
# model_fp16 = auto_convert_mixed_precision(model, feed_dict, rtol=0.01, atol=0.001, keep_io_types=True)
# onnx.save(model_fp16, "path/to/model_fp16.onnx") 

In [23]:
# Encoder fp16
import onnx
from onnxconverter_common import float16

# Encoder fp16
model = onnx.load(enc_path)
model_fp16 = float16.convert_float_to_float16(model)
onnx.save(model_fp16, "/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_fp16.onnx")


/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 2.4280380372943e-08 will be truncated to 1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -3.1657396704076746e-08 will be truncated to -1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 9.571806458552601e-08 will be truncated to 1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.6641562794461606e-08 will be truncated to -1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 7.951483027568429e-09 

In [24]:
# Decoder fp16
model = onnx.load(dec_path)
model_fp16 = float16.convert_float_to_float16(model)
onnx.save(model_fp16, "/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_fp16.onnx")

/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -4.462827973839012e-08 will be truncated to -1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 1.8726369788168995e-08 will be truncated to 1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.909895962550536e-09 will be truncated to 1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -6.181256395620949e-08 will be truncated to -1e-07
  warnings.warn(
/home/copter/miniconda3/envs/onnxoptim_arm64/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 6.790962903124864e-09

In [25]:
# Inspect post dynamic quantization
# You may see some MP models: this is common and expected, as not all operations can or should be converted to a lower precision.

inspect_model_precision("/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_fp16.onnx")
inspect_model_precision("/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_fp16.onnx")


--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder_quant_fp16.onnx ---
✅ Model appears to be a single precision type: TensorProto.FLOAT16
Model input data type: TensorProto.FLOAT16
--------------------------------------------------
--- Inspecting Model: /home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder_quant_fp16.onnx ---
✅ Model appears to be a single precision type: TensorProto.FLOAT16
Model input data type: TensorProto.FLOAT16
--------------------------------------------------
